In [ ]:
# !pip install aif360

In [ ]:
import torch
from FairReg.Regularization.EqualizedOddsLoss import EqualizedOddsLoss
from Models.logistic_regression_net import LinearClassificationNet
import random
import numpy as np
import os
from utils.model_utils import ModelUtils
from FairReg.DPLUtils.regularization_config import RegularizationConfig
from FairReg.Learning.learning_new import Learning

from utils.dataset_utils import DatasetUtils
from utils.tabular_datasets_utils import load_dutch
from scipy.io import arff
from sklearn.model_selection import train_test_split
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
import pandas as pd
from sklearn.linear_model import LogisticRegression
from utils.tabular_datasets_utils import dataset_to_numpy, load_dutch
import dill

In [ ]:
# ignore warnings

import warnings

warnings.filterwarnings("ignore")

In [ ]:
class SPATIALFairnessModule:
    ## Assumes binary values (1,0) in all the inputs, we can add extra parameters if that’s not the case
    def __init__(self, _y_true, _y_pred, _groups, _group_name):
        self.y_true = _y_true  # Binary array. Contains the true label for each sample
        self.y_pred = _y_pred  # Binary array. Contains the predictions for each sample
        self.groups = _groups  # Binary array. Defines the demogaphic group membership of each sample
        self.grouping_name = _group_name  # String. Gives a name to the grouping. Example: Gender
        _label_names = ["Y"]
        _protected_attribute_names = [_group_name]
        _favorable_label = 1
        _unfavorable_label = 0
        _df = pd.DataFrame(columns=["Y", _group_name])
        _df["Y"] = _y_true
        _df[_group_name] = _groups
        self.aif_input_data = BinaryLabelDataset(
            df=_df,
            label_names=_label_names,
            protected_attribute_names=_protected_attribute_names,
            favorable_label=_favorable_label,
            unfavorable_label=_unfavorable_label,
        )
        self.privileged_groups = [{_group_name: 1}]
        self.unprivileged_groups = [{_group_name: 0}]
        self.input_metrics = BinaryLabelDatasetMetric(
            self.aif_input_data,
            unprivileged_groups=self.unprivileged_groups,
            privileged_groups=self.privileged_groups,
        )
        self.pred_aif_data = self.aif_input_data.copy(deepcopy=True)
        self.pred_aif_data.labels = _y_pred
        self.clf_metrics = ClassificationMetric(
            self.aif_input_data,
            self.pred_aif_data,
            unprivileged_groups=self.unprivileged_groups,
            privileged_groups=self.privileged_groups,
        )

# Celeba Dataset

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
batch_size = 333

In [ ]:
train_ds, test_ds = DatasetUtils.download_dataset(
    dataset_name="celeba",
    base_path="../dataset/celeba/",
    train_csv="train_eq_odds",  # "train_original",
    test_csv="validation_eq_odds",  # "test_original",
)

In [ ]:
# train_df = pd.read_csv("../dataset/celeba/train_original.csv")

In [ ]:
# train_df, validation_df = train_test_split(train_df, test_size=0.2, random_state=seed)

# train_df.to_csv("../dataset/celeba/train_eq_odds.csv", index=False)
# validation_df.to_csv("../dataset/celeba/validation_eq_odds.csv", index=False)

In [ ]:
torch.save(train_ds, "../dataset/celeba/train_eq_odds.pt")
torch.save(test_ds, "../dataset/celeba/validation_eq_odds.pt")

In [ ]:
# torch.save(test_ds, "../dataset/celeba/validation_ds_reduced.pt")

In [ ]:
len(test_ds.targets)

In [ ]:
train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

test_loader = torch.utils.data.DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

In [ ]:
import torch.nn as nn
from torch import Tensor, nn


class CelebaNet(nn.Module):
    """This class defines the CelebaNet."""

    def __init__(
        self,
        in_channels: int = 3,
        num_classes: int = 2,
        dropout_rate: float = 0,
    ) -> None:
        """Initializes the CelebaNet network.

        Args:
        ----
            in_channels (int, optional): Number of input channels . Defaults to 3.
            num_classes (int, optional): Number of classes . Defaults to 2.
            dropout_rate (float, optional): _description_. Defaults to 0.2.
        """
        super().__init__()
        self.cnn1 = nn.Conv2d(
            in_channels,
            8,
            kernel_size=(3, 3),
            padding=(1, 1),
            stride=(1, 1),
        )
        self.cnn2 = nn.Conv2d(8, 16, kernel_size=(3, 3), padding=(1, 1), stride=(1, 1))
        self.cnn3 = nn.Conv2d(16, 32, kernel_size=(3, 3), padding=(1, 1), stride=(1, 1))
        self.fc1 = nn.Linear(2048, 2)
        self.gn_relu = nn.Sequential(
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)),
        )
        # self.dropout = nn.Dropout(dropout_rate)

    def forward(self, input_data: Tensor) -> Tensor:
        """Defines the forward pass of the network.

        Args:
            input_data (Tensor): Input data

        Returns
        -------
            Tensor: Output data
        """
        out = self.gn_relu(self.cnn1(input_data))
        out = self.gn_relu(self.cnn2(out))
        out = self.gn_relu(self.cnn3(out))
        out = out.reshape(out.size(0), -1)
        out = self.fc1(out)
        return out

# Train a model without Fairness Mitigation

In [ ]:
# Create the model that we will train, for Dutch we will use a LinearClassificationNet
# defined inside this Library in the Models/logistic_regression_net.py file
model = CelebaNet()
lr = 0.09518314591348188
optimizer = torch.optim.SGD(model.parameters(), lr=lr)
epochs = 3
seed = 42

# seed the model
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
# We don't want to use privacy in this case but we make the model private using
# the noise=0 because the code for learning the model only accept private models.
(
    private_model,
    private_optimizer,
    private_train_loader,
) = ModelUtils.create_private_model(
    model=model,
    epsilon=None,
    noise_multiplier=0,
    original_optimizer=optimizer,
    train_loader=train_loader,
    epochs=epochs,
    delta=0,
    MAX_GRAD_NORM=10000000000,  # since we just need to wrap the model without using privacy we use a high value here
    batch_size=batch_size,
)

In [ ]:
model.to("cuda")
private_model.to("cuda")

In [ ]:
train_parameters = RegularizationConfig(
    epochs=epochs,
    device="cuda",
    batch_size=batch_size,
    seed=seed,
    optimizer="adam",
    regularization=False,
)

In [ ]:
def compute_violation_with_argmax(
    sensitive_attribute_list: torch.tensor,
    analysis_dict: dict,
    y_pred: torch.tensor,
):
    return max(
        # FPR
        abs(
            (len(analysis_dict[(0, 1, 1)]) / (len(analysis_dict[(0, 1, 1)]) + len(analysis_dict[(0, 0, 1)])))
            - (len(analysis_dict[(0, 1, 0)]) / (len(analysis_dict[(0, 1, 0)]) + len(analysis_dict[(0, 0, 0)])))
        ),
        # TPR
        abs(
            (len(analysis_dict[(1, 1, 1)]) / (len(analysis_dict[(1, 1, 1)]) + len(analysis_dict[(1, 0, 1)])))
            - (len(analysis_dict[(1, 1, 0)]) / (len(analysis_dict[(1, 1, 0)]) + len(analysis_dict[(1, 0, 0)])))
        ),
    )


def compute_error_rate_difference(
    sensitive_attribute_list: torch.tensor,
    analysis_dict: dict,
    y_pred: torch.tensor,
):
    return abs(
        (
            (len(analysis_dict[(0, 1, 0)]) + len(analysis_dict[(1, 0, 0)]))
            / (
                len(analysis_dict[(0, 1, 0)])
                + len(analysis_dict[(1, 0, 0)])
                + len(analysis_dict[(1, 1, 0)])
                + len(analysis_dict[(0, 0, 0)])
            )
        )
        - (
            (len(analysis_dict[(0, 1, 1)]) + len(analysis_dict[(1, 0, 1)]))
            / (
                len(analysis_dict[(0, 1, 1)])
                + len(analysis_dict[(1, 0, 1)])
                + len(analysis_dict[(1, 1, 1)])
                + len(analysis_dict[(0, 0, 1)])
            )
        )
    )

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

# from .DPL.DPLUtilsutils import Utils


class MyEqualizedOddsLoss(nn.Module):
    def __init__(self, weight=None, size_average=True, estimation=0.5) -> None:
        """Initialization of the regularization loss."""
        super().__init__()
        self.estimation = estimation

    def forward(
        self,
        sensitive_attribute_list: torch.tensor,
        device: torch.device,
        predictions: torch.tensor,
        true_targets: torch.tensor,
        possible_sensitive_attributes: list,
        possible_targets: list,
        average_probabilities: dict = None,
    ) -> torch.tensor:
        fairness_violations = []
        # We compute the softmax of the predictions. We do this because
        # we can't use the argmax function on the nn output,
        # because we need differentiable results
        softmax_ = F.softmax(predictions, dim=1)

        # convert the list of sensitive attributes to a tensor and move it to the device
        sensitive_attribute_list = torch.tensor([int(item) for item in sensitive_attribute_list])
        sensitive_attribute_list = sensitive_attribute_list.to(device)
        true_targets = torch.tensor([int(item) for item in true_targets]).to(device)

        # We compute the argmax of the predictions, this is used to count
        # the number of samples for each class that are predicted with one class
        # or with the other.
        predictions_argmax = torch.argmax(torch.tensor(predictions), dim=1).to(device)
        # we convert the possible targets and the possible sensitive attributes to a list
        # just to be sure that the values are integers
        possible_targets = [int(item) for item in possible_targets]
        possible_sensitive_attributes = [int(item) for item in possible_sensitive_attributes]

        analysis_dict = {}

        sensitive_attribute_list = [1 if item == 1.0 else 0 for item in sensitive_attribute_list]

        for index, y, prediction, group in zip(
            list(range(len(predictions))),
            true_targets,
            predictions_argmax,
            sensitive_attribute_list,
        ):
            prediction = int(prediction.item())
            y = int(y.item())
            if (y, prediction, group) not in analysis_dict:
                analysis_dict[(y, prediction, group)] = []
            analysis_dict[(y, prediction, group)].append(index)

        fp_male = torch.sum(softmax_[analysis_dict[(0, 1, 1)]][:, 1])
        fp_female = torch.sum(softmax_[analysis_dict[(0, 1, 0)]][:, 1])
        tn_male = torch.sum(softmax_[analysis_dict[(0, 0, 1)]][:, 0])
        tn_female = torch.sum(softmax_[analysis_dict[(0, 0, 0)]][:, 0])

        tp_male = torch.sum(softmax_[analysis_dict[(1, 1, 1)]][:, 1])
        tp_female = torch.sum(softmax_[analysis_dict[(1, 1, 0)]][:, 1])
        fn_male = torch.sum(softmax_[analysis_dict[(1, 0, 1)]][:, 0])
        fn_female = torch.sum(softmax_[analysis_dict[(1, 0, 0)]][:, 0])

        print(
            "fp_male: ",
            fp_male,
            "fp_female: ",
            fp_female,
            "tn_male: ",
            tn_male,
            "tn_female: ",
            tn_female,
        )
        print(
            "tp_male: ",
            tp_male,
            "tp_female: ",
            tp_female,
            "fn_male: ",
            fn_male,
            "fn_female: ",
            fn_female,
        )

        fairness_violations.append(
            max(
                # FPR
                abs((fp_male / (fp_male + tn_male)) - (fp_female / (fp_female + tn_female))),
                # TPR
                abs((tp_male / (tp_male + fn_male)) - (tp_female / (tp_female + fn_female))),
            )
        )

        print(fairness_violations)

        fairness_violations_ = [item.item() if isinstance(item, torch.Tensor) else item for item in fairness_violations]

        # We get the index of the maximum violation term. Then we create a mask with
        # all zeros and we set to 1 the element at the index we found. We use this mask
        # to sum the violation terms and we return the result. This was needed because
        # when we started to work on this project we discovered that without this
        # some of the gradients were not computed correctly. I would not remove it
        # even if I'm not sure that it is needed anymore.
        index = fairness_violations_.index(max(fairness_violations_))
        fairness_violations = torch.stack(fairness_violations)
        mask = torch.full((fairness_violations.shape[0],), 0, dtype=torch.float32).to(device)
        mask[index] = 1
        res = torch.sum(mask * fairness_violations)

        return res

    def violation_with_dataset(
        self,
        model: torch.nn.Module,
        dataset: torch.utils.data.DataLoader,
        average_probabilities: dict,
        device: torch.device,
    ) -> torch.tensor:
        predictions = torch.tensor([]).to(device)
        sensitive_attribute_list = torch.tensor([]).to(device)
        targets = []
        model.eval()
        with torch.no_grad():
            for images, sensitive_attributes, target, _, _ in dataset:
                images = images.to(device)
                target = target.to(device)

                output = model(images)

                predictions = torch.cat((predictions, output), 0)
                sensitive_attribute_list = torch.cat((sensitive_attribute_list, sensitive_attributes.to(device)), 0)
                targets += target.tolist()

        sensitive_attributes = list({item.item() for item in sensitive_attribute_list})
        target_list = list(set(targets))

        # now we just call the forward function with the "fake" predictions and the sensitive
        # attribute list we computed
        return self.forward(
            sensitive_attribute_list=sensitive_attribute_list,
            device=device,
            predictions=predictions,
            true_targets=np.array(targets),
            possible_sensitive_attributes=sensitive_attributes,
            possible_targets=target_list,
            average_probabilities=average_probabilities,
        )

In [ ]:
for epoch in range(0, epochs):
    print("OK")
    # Now we can train the model. First of all we will train a model without any
    # fairness mitigation
    results = Learning.train_private_model(
        train_parameters=train_parameters,
        model=private_model,
        model_regularization=None,
        optimizer=private_optimizer,
        optimizer_regularization=None,
        train_loader=train_loader,
        test_loader=test_loader,
        average_probabilities=None,
        current_epoch=epoch,
    )
    print("OK")
    (
        _,
        accuracy,
        _,
        _,
        _,
        max_disparity_test,
        y_true,
        y_pred,
        sensitive_attributes,
        real_indexes,
        _,
    ) = Learning.test(
        model=private_model,
        test_loader=train_loader,
        train_parameters=train_parameters,
        current_epoch=epochs,
    )
    sensitive_attributes = [1 if item == 1.0 else 0 for item in sensitive_attributes]

    analysis_dict = {}
    for index, y, prediction, group in zip(real_indexes, y_true, y_pred, sensitive_attributes):
        if (y, prediction, group) not in analysis_dict:
            analysis_dict[(y, prediction, group)] = []
        analysis_dict[(y, prediction, group)].append(index)

    for key in analysis_dict:
        print(f"{key}: {len(analysis_dict[key])}")

    fairness_module = SPATIALFairnessModule(
        _y_true=np.array(y_true),
        _y_pred=np.array(y_pred),
        _groups=np.array(sensitive_attributes),
        _group_name="Gender",
    )

    print(
        "equalized_odds_difference: ",
        fairness_module.clf_metrics.equalized_odds_difference(),
    )
    print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())

    print(
        "My Equalized Odds: ",
        compute_violation_with_argmax(
            sensitive_attribute_list=np.array(sensitive_attributes),
            analysis_dict=analysis_dict,
            y_pred=np.array(y_pred),
        ),
    )

    print(
        "My EO: ",
        MyEqualizedOddsLoss().violation_with_dataset(private_model, train_loader, None, "cuda"),
    )

    print(
        f"Epoch {epoch} - Train accuracy {results['Train Accuracy']} - Train Loss {results['Train Loss']} - Equalized Odds Difference {max_disparity_test}"
    )

In [ ]:
(
    _,
    accuracy,
    _,
    _,
    _,
    max_disparity_test,
    y_true_test,
    y_pred_test,
    sensitive_attributes_test,
    real_indexes_test,
    _,
) = Learning.test(
    model=private_model,
    test_loader=test_loader,
    train_parameters=train_parameters,
    current_epoch=epochs,
)
sensitive_attributes_test = [1 if item == 1.0 else 0 for item in sensitive_attributes_test]
analysis_dict_test = {}
for index, y, prediction, group in zip(real_indexes_test, y_true_test, y_pred_test, sensitive_attributes_test):
    if (y, prediction, group) not in analysis_dict_test:
        analysis_dict_test[(y, prediction, group)] = []
    analysis_dict_test[(y, prediction, group)].append(index)

for key in analysis_dict_test:
    print(f"{key}: {len(analysis_dict_test[key])}")

print(f"Test accuracy {accuracy} - Equalized Odds Difference {max_disparity_test}")

fairness_module = SPATIALFairnessModule(
    _y_true=np.array(y_true_test),
    _y_pred=np.array(y_pred_test),
    _groups=np.array(sensitive_attributes_test),
    _group_name="Gender",
)

print(
    "My Equalized Odds: ",
    compute_violation_with_argmax(
        sensitive_attribute_list=np.array(sensitive_attributes_test),
        analysis_dict=analysis_dict_test,
        y_pred=np.array(y_pred_test),
    ),
)

print(
    "My Error Rate Difference: ",
    compute_error_rate_difference(
        sensitive_attribute_list=np.array(sensitive_attributes_test),
        analysis_dict=analysis_dict_test,
        y_pred=np.array(y_pred_test),
    ),
)

print(
    "equalized_odds_difference: ",
    fairness_module.clf_metrics.equalized_odds_difference(),
)
print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())

In [ ]:
for key in analysis_dict_test:
    print(f"{key}: {len(analysis_dict_test[key])}")

In [ ]:
import copy

analysis_dict_test_copy = copy.deepcopy(analysis_dict)

In [ ]:
for key in analysis_dict_test_copy:
    print(f"{key}: {len(analysis_dict_test_copy[key])}")

In [ ]:
analysis_dict_test_copy[(1, 1, 1)] = analysis_dict_test_copy[(1, 1, 1)][
    0 : len(analysis_dict_test_copy[(1, 1, 1)]) // 3
]

In [ ]:
for key in analysis_dict_test_copy:
    print(f"{key}: {len(analysis_dict_test_copy[key])}")

In [ ]:
compute_violation_with_argmax(
    sensitive_attribute_list=np.array(sensitive_attributes_test),
    analysis_dict=analysis_dict_test_copy,
    y_pred=np.array(y_pred_test),
)

# Removing TP

In [119]:
# with open("./analysis_dict.pkl", "wb") as file:
#     dill.dump(analysis_dict, file)

# with open("./analysis_dict_test.pkl", "wb") as file:
#     dill.dump(analysis_dict_test, file)

with open("./analysis_dict.pkl", "rb") as file:
    # dill.dump(indexes_to_remove, file)
    analysis_dict = dill.load(file)

with open("./analysis_dict_test.pkl", "rb") as file:
    # dill.dump(indexes_to_remove, file)
    analysis_dict_test = dill.load(file)

In [120]:
# train_ds_tmp, test_ds_tmp = DatasetUtils.download_dataset(
#     dataset_name="celeba",
#     base_path="../dataset/celeba/",
#     train_csv="train_original",
#     test_csv="test_original",
# )

In [121]:
import copy

train_ds = copy.copy(train_ds_tmp)
test_ds = copy.copy(test_ds_tmp)

In [122]:
print(len(train_ds.samples))

162417


In [123]:
from FairReg.Regularization.RegularizationLoss import RegularizationLoss

RegularizationLoss().compute_violation_with_argmax(
    predictions_argmax=train_ds.targets,
    sensitive_attribute_list=train_ds.sensitive_attributes,
    current_target=1,
    current_sensitive_feature=1,
)

print(
    "My Equalized Odds: ",
    compute_violation_with_argmax(
        sensitive_attribute_list=train_ds.sensitive_attributes,
        analysis_dict=analysis_dict,
        y_pred=train_ds.targets,
    ),
)

My Equalized Odds:  0.056893521688172854


In [124]:
indexes_to_remove = analysis_dict[(1, 1, 1)][0 : int(len(analysis_dict[(1, 1, 1)]) * (2 / 3))]
indexes_to_remove_test = analysis_dict_test[(1, 1, 1)][0 : int(len(analysis_dict_test[(1, 1, 1)]) * (2 / 3))]

In [125]:
# remove indexes from train_ds
train_ds.samples = np.delete(np.array(train_ds.samples), indexes_to_remove, axis=0)
train_ds.targets = np.delete(np.array(train_ds.targets), indexes_to_remove, axis=0)
train_ds.sensitive_attributes = np.delete(np.array(train_ds.sensitive_attributes), indexes_to_remove, axis=0)

In [126]:
# create the validation set
indexes_validation = np.random.choice(np.array(train_ds.samples).shape[0], 15000, replace=False)
validation_samples = np.array(train_ds.samples)[indexes_validation]
validation_targets = np.array(train_ds.targets)[indexes_validation]
validation_sensitive_attributes = np.array(train_ds.sensitive_attributes)[indexes_validation]

train_samples = np.delete(np.array(train_ds.samples), indexes_validation, axis=0)
train_targets = np.delete(np.array(train_ds.targets), indexes_validation, axis=0)
train_sensitive_attributes = np.delete(np.array(train_ds.sensitive_attributes), indexes_validation, axis=0)

from utils.celeba import CelebaDatasetData
from torchvision import transforms

transform = transforms.Compose(
    [
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ],
)

image_path = "../dataset/celeba/img_align_celeba"

celeba_train = CelebaDatasetData(
    samples=train_samples,
    sensitive_attributes=train_sensitive_attributes,
    targets=train_targets,
    image_path=image_path,
    transform=transform,
)

celeba_validation = CelebaDatasetData(
    samples=validation_samples,
    sensitive_attributes=validation_sensitive_attributes,
    targets=validation_targets,
    image_path=image_path,
    transform=transform,
)

In [127]:
# remove indexes from test_ds
test_ds_samples = np.delete(test_ds.samples, indexes_to_remove_test, axis=0)
test_ds_targets = np.delete(test_ds.targets, indexes_to_remove_test, axis=0)
test_ds_sensitive_attributes = np.delete(test_ds.sensitive_attributes, indexes_to_remove_test, axis=0)

celeba_test = CelebaDatasetData(
    samples=test_ds_samples,
    sensitive_attributes=test_ds_sensitive_attributes,
    targets=test_ds_targets,
    image_path=image_path,
    transform=transform,
)

In [128]:
len(analysis_dict[(1, 1, 1)])

11769

In [129]:
int(len(analysis_dict[(1, 1, 1)]) // (2 / 3))

17653

In [130]:
analysis_dict[(1, 1, 1)] = analysis_dict[(1, 1, 1)][int(len(analysis_dict[(1, 1, 1)]) * (2 / 3)) :]

print(
    "My Equalized Odds After: ",
    compute_violation_with_argmax(
        sensitive_attribute_list=celeba_train.sensitive_attributes,
        analysis_dict=analysis_dict,
        y_pred=celeba_train.targets,
    ),
)

My Equalized Odds After:  0.2500493853261092


In [132]:
import torch

torch.save(celeba_train, "../dataset/celeba/new_celeba_train.pt")
torch.save(celeba_test, "../dataset/celeba/new_celeba_test.pt")
torch.save(celeba_validation, "../dataset/celeba/new_celeba_validation.pt")

In [ ]:
from FairReg.Regularization.RegularizationLoss import RegularizationLoss

RegularizationLoss().compute_violation_with_argmax(
    predictions_argmax=train_ds.targets,
    sensitive_attribute_list=train_ds.sensitive_attributes,
    current_target=1,
    current_sensitive_feature=1,
)

In [ ]:
# torch.save(train_ds, "../dataset/celeba/train_ds.pt")
# torch.save(test_ds, "../dataset/celeba/test_ds.pt")

# # train_ds = torch.load("../dataset/dutch/train_ds.pt")
# # test_ds = torch.load("../dataset/dutch/test_ds.pt")

In [ ]:
train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

test_loader = torch.utils.data.DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

In [ ]:
# Create the model that we will train, for Dutch we will use a LinearClassificationNet
# defined inside this Library in the Models/logistic_regression_net.py file
model = LinearClassificationNet()
lr = 0.019925917176300392
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
epochs = 5
seed = 42

# seed the model
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
# We don't want to use privacy in this case but we make the model private using
# the noise=0 because the code for learning the model only accept private models.
(
    private_model,
    private_optimizer,
    private_train_loader,
) = ModelUtils.create_private_model(
    model=model,
    epsilon=None,
    noise_multiplier=0,
    original_optimizer=optimizer,
    train_loader=train_loader,
    epochs=epochs,
    delta=0,
    MAX_GRAD_NORM=10000000000,  # since we just need to wrap the model without using privacy we use a high value here
    batch_size=batch_size,
)

In [ ]:
model.to("cuda")
private_model.to("cuda")

In [ ]:
train_parameters = RegularizationConfig(
    epochs=epochs,
    device="cuda",
    batch_size=batch_size,
    seed=seed,
    optimizer="adam",
    regularization=False,
)

In [ ]:
for epoch in range(0, epochs):
    # Now we can train the model. First of all we will train a model without any
    # fairness mitigation
    results = Learning.train_private_model(
        train_parameters=train_parameters,
        model=private_model,
        model_regularization=None,
        optimizer=private_optimizer,
        optimizer_regularization=None,
        train_loader=train_loader,
        test_loader=test_loader,
        average_probabilities=None,
        current_epoch=epoch,
    )
    (
        _,
        accuracy,
        _,
        _,
        _,
        max_disparity_test,
        y_true,
        y_pred,
        sensitive_attributes,
        real_indexes,
        _,
    ) = Learning.test(
        model=private_model,
        test_loader=train_loader,
        train_parameters=train_parameters,
        current_epoch=epochs,
    )

    analysis_dict = {}
    for index, y, prediction, group in zip(real_indexes, y_true, y_pred, sensitive_attributes):
        if (y, prediction, group) not in analysis_dict:
            analysis_dict[(y, prediction, group)] = []
        analysis_dict[(y, prediction, group)].append(index)

    for key in analysis_dict:
        print(f"{key}: {len(analysis_dict[key])}")

    fairness_module = SPATIALFairnessModule(
        _y_true=np.array(y_true),
        _y_pred=np.array(y_pred),
        _groups=np.array(sensitive_attributes),
        _group_name="Gender",
    )

    print(
        "equalized_odds_difference: ",
        fairness_module.clf_metrics.equalized_odds_difference(),
    )
    print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())

    print(
        f"Epoch {epoch} - Train accuracy {results['Train Accuracy']} - Train Loss {results['Train Loss']} - Equalised Odds Difference {max_disparity_test}"
    )

In [ ]:
(
    _,
    accuracy,
    _,
    _,
    _,
    max_disparity_test,
    y_true_test,
    y_pred_test,
    sensitive_attributes_test,
    real_indexes_test,
    _,
) = Learning.test(
    model=private_model,
    test_loader=test_loader,
    train_parameters=train_parameters,
    current_epoch=epochs,
)

analysis_dict_test = {}
for index, y, prediction, group in zip(real_indexes_test, y_true_test, y_pred_test, sensitive_attributes_test):
    if (y, prediction, group) not in analysis_dict_test:
        analysis_dict_test[(y, prediction, group)] = []
    analysis_dict_test[(y, prediction, group)].append(index)

for key in analysis_dict_test:
    print(f"{key}: {len(analysis_dict_test[key])}")

print(f"Test accuracy {accuracy} - Equalized Odds Difference {max_disparity_test}")

fairness_module = SPATIALFairnessModule(
    _y_true=np.array(y_true_test),
    _y_pred=np.array(y_pred_test),
    _groups=np.array(sensitive_attributes_test),
    _group_name="Gender",
)

print(
    "My Equalized Odds: ",
    compute_violation_with_argmax(
        sensitive_attribute_list=np.array(sensitive_attributes_test),
        analysis_dict=analysis_dict_test,
        y_pred=np.array(y_pred_test),
    ),
)

print(
    "equalized_odds_difference: ",
    fairness_module.clf_metrics.equalized_odds_difference(),
)
print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())

# Baseline Celeba

In [ ]:
batch_size = 256
lr = 0.1
epochs = 10
seed = 42

In [ ]:
import torch

celeba_train = torch.load("../dataset/celeba/new_celeba_train.pt")
celeba_test = torch.load("../dataset/celeba/new_celeba_test.pt")

In [ ]:
train_loader = torch.utils.data.DataLoader(
    celeba_train,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

test_loader = torch.utils.data.DataLoader(
    celeba_test,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

In [ ]:
model = CelebaNet()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
model.to("cuda")

In [ ]:
(
    private_model,
    private_optimizer,
    _,
) = ModelUtils.create_private_model(
    model=model,
    epsilon=None,
    noise_multiplier=0,
    original_optimizer=optimizer,
    train_loader=train_loader,
    epochs=epochs,
    delta=0,
    MAX_GRAD_NORM=10000000000,  # since we just need to wrap the model without using privacy we use a high value here
    batch_size=batch_size,
)

In [ ]:
train_parameters = RegularizationConfig(
    epochs=epochs,
    device="cuda",
    batch_size=batch_size,
    seed=seed,
    optimizer="adam",
)

In [ ]:
for epoch in range(0, 10):
    results = Learning.train_private_model(
        train_parameters=train_parameters,
        model=private_model,
        model_regularization=None,
        optimizer=private_optimizer,
        optimizer_regularization=None,
        train_loader=train_loader,
        test_loader=test_loader,
        average_probabilities=None,
        current_epoch=epoch,
        epoch=epoch,
    )
    (
        _,
        accuracy,
        _,
        _,
        _,
        max_disparity_test,
        y_true,
        y_pred,
        sensitive_attributes,
        _,
        _,
    ) = Learning.test(
        model=private_model,
        test_loader=train_loader,
        train_parameters=train_parameters,
        current_epoch=epochs,
    )

    fairness_module = SPATIALFairnessModule(
        _y_true=np.array(y_true),
        _y_pred=np.array(y_pred),
        _groups=np.array(sensitive_attributes),
        _group_name="Gender",
    )

    print(
        "equalized_odds_difference: ",
        fairness_module.clf_metrics.equalized_odds_difference(),
    )
    print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())

    print(
        f"Epoch {epoch} - Train accuracy {results['Train Accuracy']} - Train Loss {results['Train Loss']} - Equalized Odds Difference {max_disparity_test} - Regularization Loss {results['Train Loss + Regularizaion']}"
    )

# Train a model with Fairness Mitigation

In [ ]:
batch_size = 333
# torch.save(train_ds, "../dataset/dutch/train_ds.pt")
# torch.save(test_ds, "../dataset/dutch/test_ds.pt")

train_ds = torch.load("../dataset/dutch/train_ds.pt")
test_ds = torch.load("../dataset/dutch/test_ds.pt")

In [ ]:
len(train_ds.samples)

In [ ]:
# sample 8000 samples from the train_ds.samples
indexes = np.random.choice(train_ds.samples.shape[0], 8000, replace=False)

In [ ]:
from utils.dutch import TabularDataset

samples = train_ds.samples[indexes]
targets = train_ds.targets[indexes]
sensitive_features = train_ds.sensitive_features[indexes]

validation_ds = TabularDataset(samples, targets, sensitive_features)

In [ ]:
train_ds.samples = np.delete(train_ds.samples, indexes, axis=0)
train_ds.targets = np.delete(train_ds.targets, indexes, axis=0)
train_ds.sensitive_features = np.delete(train_ds.sensitive_features, indexes, axis=0)
train_ds.indexes = np.delete(train_ds.indexes, indexes, axis=0)

In [ ]:
len(train_ds.samples)

In [ ]:
validation_ds

In [ ]:
train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

test_loader = torch.utils.data.DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

In [ ]:
# Create the model that we will train, for Dutch we will use a LinearClassificationNet
# defined inside this Library in the Models/logistic_regression_net.py file
model = LinearClassificationNet()
model_regularization = LinearClassificationNet()

lr = 0.019925917176300392
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
optimizer_regularization = torch.optim.SGD(model_regularization.parameters(), lr=lr)

epochs = 40
seed = 42

# seed the model
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
# We don't want to use privacy in this case but we make the model private using
# the noise=0 because the code for learning the model only accept private models.
(
    private_model,
    private_optimizer,
    private_train_loader,
) = ModelUtils.create_private_model(
    model=model,
    epsilon=None,
    noise_multiplier=0,
    original_optimizer=optimizer,
    train_loader=train_loader,
    epochs=epochs,
    delta=0,
    MAX_GRAD_NORM=10000000000,  # since we just need to wrap the model without using privacy we use a high value here
    batch_size=batch_size,
)

In [ ]:
# We don't want to use privacy in this case but we make the model private using
# the noise=0 because the code for learning the model only accept private models.

(
    private_model_regularization,
    private_optimizer_regularization,
    _,
) = ModelUtils.create_private_model(
    model=model_regularization,
    epsilon=None,
    noise_multiplier=0,
    original_optimizer=optimizer_regularization,
    train_loader=train_loader,
    epochs=epochs,
    delta=0,
    MAX_GRAD_NORM=10000000000,  # since we just need to wrap the model without using privacy we use a high value here
    batch_size=batch_size,
)

In [ ]:
model.to("cuda")
private_model.to("cuda")

In [ ]:
train_parameters = RegularizationConfig(
    epochs=epochs,
    device="cuda",
    batch_size=batch_size,
    seed=seed,
    regularization=True,
    target=0.05,
    regularization_mode="fixed",
    regularization_lambda=0.9,
    optimizer="adam",
)

In [ ]:
for epoch in range(0, epochs):
    # Now we can train the model. First of all we will train a model without any
    # fairness mitigation
    results = Learning.train_private_model(
        train_parameters=train_parameters,
        model=private_model,
        model_regularization=private_model_regularization,
        optimizer=private_optimizer,
        optimizer_regularization=private_optimizer_regularization,
        train_loader=train_loader,
        test_loader=test_loader,
        average_probabilities=None,
        current_epoch=epoch,
    )
    (
        _,
        accuracy,
        _,
        _,
        _,
        max_disparity_test,
        y_true,
        y_pred,
        sensitive_attributes,
        _,
        _,
    ) = Learning.test(
        model=private_model,
        test_loader=train_loader,
        train_parameters=train_parameters,
        current_epoch=epochs,
    )

    fairness_module = SPATIALFairnessModule(
        _y_true=np.array(y_true),
        _y_pred=np.array(y_pred),
        _groups=np.array(sensitive_attributes),
        _group_name="Gender",
    )

    print(
        "equalized_odds_difference: ",
        fairness_module.clf_metrics.equalized_odds_difference(),
    )
    print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())

    print(
        f"Epoch {epoch} - Train accuracy {results['Train Accuracy']} - Train Loss {results['Train Loss']} - Equalized Odds Difference {max_disparity_test} - Regularization Loss {results['Train Loss + Regularizaion']}"
    )

In [ ]:
# We can test the trained model on the test dataset to understand if the model is less unfair than before
(
    _,
    accuracy,
    _,
    _,
    _,
    _,
    y_true_test,
    y_pred_test,
    sensitive_attribute_test,
) = Learning.test(
    model=private_model,
    test_loader=test_loader,
    train_parameters=train_parameters,
    current_epoch=epochs,
)

_, _, _, _, _, max_disparity_test, _, _, _ = Learning.test(
    model=private_model,
    test_loader=test_loader,
    train_parameters=train_parameters,
    current_epoch=epochs,
)

print(f"Test accuracy {accuracy} - Average Predictive Value Difference  {max_disparity_test}")

fairness_module = SPATIALFairnessModule(
    _y_true=np.array(y_true_test),
    _y_pred=np.array(y_pred_test),
    _groups=np.array(sensitive_attribute_test),
    _group_name="Gender",
)

print(
    "equalized_odds_difference: ",
    fairness_module.clf_metrics.equalized_odds_difference(),
)
print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())